<a href="https://colab.research.google.com/github/SakshyamAryal077/FOUNDATION-MODEL-DOWNSTREAM-PREDICTION/blob/main/CeilPrep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")

# ============================================================
# PATHS
# ============================================================

ROOT = Path("/content/drive/MyDrive/ECG_Project")

PROCESSED = (
    ROOT /
    "results" /
    "preprocessing" /
    "processed"
)

TRAIN_CSV = PROCESSED / "train_omi.csv"
VAL_CSV   = PROCESSED / "validation_omi.csv"

OUT = (
    ROOT /
    "results" /
    "ceiling_analysis"
)

OUT.mkdir(
    parents=True,
    exist_ok=True
)

# ============================================================
# LOAD
# ============================================================

train = pd.read_csv(TRAIN_CSV)
val   = pd.read_csv(VAL_CSV)

train["split"] = "train"
val["split"]   = "validation"

df = pd.concat(
    [train, val],
    ignore_index=True
)

print("Total ECGs:", len(df))
print("Train:", len(train))
print("Validation:", len(val))

# ============================================================
# SHOW AVAILABLE COLUMNS
# ============================================================

print("\n================ AVAILABLE COLUMNS ================")

for c in df.columns:
    print(c)

# ============================================================
# BASIC QUALITY CHECKS
# ============================================================

print("\n================ QUALITY CHECK ================")

if "record_id" in df.columns:
    print("Unique records:", df["record_id"].nunique())
    print("Duplicate record IDs:", df["record_id"].duplicated().sum())

if "Patient_id" in df.columns:
    print("Unique patients:", df["Patient_id"].nunique())

    train_patients = set(
        train["Patient_id"].astype(str)
    )

    val_patients = set(
        val["Patient_id"].astype(str)
    )

    overlap = train_patients & val_patients

    print(
        "Patient overlap train/validation:",
        len(overlap)
    )

# ============================================================
# IMPORTANT LABELS
# ============================================================

possible_labels = [
    "OMI",
    "omi",
    "STEMI",
    "NSTEMI",
    "LM",
    "PLAD",
    "MLAD",
    "DLAD",
    "DB",
    "PLCX",
    "MLCX",
    "DLCX",
    "OM",
    "PRCA",
    "MRCA",
    "DRCA"
]

existing_labels = [
    c for c in possible_labels
    if c in df.columns
]

print("\n================ LABEL DISTRIBUTION ================")

for col in existing_labels:

    x = pd.to_numeric(
        df[col],
        errors="coerce"
    )

    positives = (x > 0).sum()
    valid = x.notna().sum()

    print(
        f"{col:10s} "
        f"positive={positives:5d} "
        f"valid={valid:5d} "
        f"prevalence={positives/max(valid,1):.4f}"
    )

# ============================================================
# BUILD VESSEL-LEVEL LABELS
# ============================================================

def any_positive(columns):

    available = [
        c for c in columns
        if c in df.columns
    ]

    if not available:
        return np.zeros(
            len(df),
            dtype=int
        )

    temp = (
        df[available]
        .apply(
            pd.to_numeric,
            errors="coerce"
        )
        .fillna(0)
    )

    return (
        temp.max(axis=1) > 0
    ).astype(int)


if "LM" in df.columns:
    df["LM_GROUP"] = (
        pd.to_numeric(
            df["LM"],
            errors="coerce"
        )
        .fillna(0)
        .gt(0)
        .astype(int)
    )

df["LAD_GROUP"] = any_positive(
    ["PLAD", "MLAD", "DLAD", "DB"]
)

df["LCX_GROUP"] = any_positive(
    ["PLCX", "MLCX", "DLCX", "OM"]
)

df["RCA_GROUP"] = any_positive(
    ["PRCA", "MRCA", "DRCA"]
)

print("\n================ VESSEL GROUPS ================")

for col in [
    "LM_GROUP",
    "LAD_GROUP",
    "LCX_GROUP",
    "RCA_GROUP"
]:

    if col in df.columns:

        print(
            f"{col:10s}",
            int(df[col].sum()),
            f"({df[col].mean()*100:.2f}%)"
        )

# ============================================================
# MISSING DATA
# ============================================================

print("\n================ MOST MISSING COLUMNS ================")

missing = (
    df.isna()
    .mean()
    .sort_values(
        ascending=False
    )
)

print(
    missing.head(25)
)

# ============================================================
# SAVE MASTER TABLE
# ============================================================

MASTER_PATH = (
    OUT /
    "master_analysis_table.csv"
)

df.to_csv(
    MASTER_PATH,
    index=False
)

print("\nSaved:")
print(MASTER_PATH)

Mounted at /content/drive
Total ECGs: 17958
Train: 15273
Validation: 2685

================ AVAILABLE COLUMNS ================
record_id
Patient_id
OMI
STEMI
NSTEMI
CTO
LM
PLAD
MLAD
DLAD
DB
PLCX
MLCX
DLCX
OM
PRCA
MRCA
DRCA
split

================ QUALITY CHECK ================
Unique records: 17958
Duplicate record IDs: 0
Unique patients: 17016
Patient overlap train/validation: 0

================ LABEL DISTRIBUTION ================
OMI        positive= 1151 valid=17958 prevalence=0.0641
STEMI      positive= 1442 valid=17958 prevalence=0.0803
NSTEMI     positive= 1235 valid=17958 prevalence=0.0688
LM         positive=    9 valid=17958 prevalence=0.0005
PLAD       positive=  561 valid=17958 prevalence=0.0312
MLAD       positive=  224 valid=17958 prevalence=0.0125
DLAD       positive=   28 valid=17958 prevalence=0.0016
DB         positive=   62 valid=17958 prevalence=0.0035
PLCX       positive=  267 valid=17958 prevalence=0.0149
MLCX       positive=  156 valid=17958 prevalence=0.0087
DLC

In [ ]:
# ============================================================
# STEP 2 — OMI PHENOTYPE / LABEL HETEROGENEITY ANALYSIS
# ============================================================

import pandas as pd
import numpy as np

# df should already exist from Step 1

omi = df[df["OMI"] == 1].copy()

print("================================================")
print("OMI OVERVIEW")
print("================================================")

print("Total OMI:", len(omi))

print(
    "Train OMI:",
    len(omi[omi["split"] == "train"])
)

print(
    "Validation OMI:",
    len(omi[omi["split"] == "validation"])
)

print("\n================================================")
print("OMI vs STEMI / NSTEMI")
print("================================================")

groups = {
    "OMI + STEMI only":
        (
            (omi["STEMI"] == 1)
            &
            (omi["NSTEMI"] == 0)
        ),

    "OMI + NSTEMI only":
        (
            (omi["NSTEMI"] == 1)
            &
            (omi["STEMI"] == 0)
        ),

    "OMI + STEMI + NSTEMI":
        (
            (omi["STEMI"] == 1)
            &
            (omi["NSTEMI"] == 1)
        ),

    "OMI neither STEMI nor NSTEMI":
        (
            (omi["STEMI"] == 0)
            &
            (omi["NSTEMI"] == 0)
        )
}

for name, mask in groups.items():

    n = mask.sum()

    print(
        f"{name:30s}: "
        f"{n:4d} "
        f"({100*n/len(omi):5.2f}%)"
    )


print("\n================================================")
print("OMI BY MAJOR VESSEL")
print("================================================")

for vessel in [
    "LM_GROUP",
    "LAD_GROUP",
    "LCX_GROUP",
    "RCA_GROUP"
]:

    n = int(
        omi[vessel].sum()
    )

    print(
        f"{vessel:10s}: "
        f"{n:4d} "
        f"({100*n/len(omi):5.2f}%)"
    )


# ============================================================
# NUMBER OF POSITIVE VESSEL GROUPS PER OMI
# ============================================================

vessel_cols = [
    "LM_GROUP",
    "LAD_GROUP",
    "LCX_GROUP",
    "RCA_GROUP"
]

omi["n_vessel_groups"] = (
    omi[vessel_cols]
    .sum(axis=1)
)

print("\n================================================")
print("NUMBER OF VESSEL GROUPS PER OMI CASE")
print("================================================")

print(
    omi["n_vessel_groups"]
    .value_counts()
    .sort_index()
)


# ============================================================
# OMI WITH NO VESSEL LABEL
# ============================================================

no_vessel = (
    omi["n_vessel_groups"] == 0
)

print(
    "\nOMI with NO major vessel label:",
    no_vessel.sum(),
    f"({100*no_vessel.mean():.2f}%)"
)


# ============================================================
# CTO OVERLAP
# ============================================================

if "CTO" in omi.columns:

    cto_n = (
        omi["CTO"] == 1
    ).sum()

    print(
        "\nOMI + CTO:",
        cto_n,
        f"({100*cto_n/len(omi):.2f}%)"
    )


# ============================================================
# TRAIN vs VALIDATION PHENOTYPE COUNTS
# ============================================================

print("\n================================================")
print("TRAIN / VALIDATION OMI PHENOTYPES")
print("================================================")

for split_name in [
    "train",
    "validation"
]:

    temp = omi[
        omi["split"] == split_name
    ]

    print(f"\n--- {split_name.upper()} ---")
    print("OMI:", len(temp))

    print(
        "STEMI:",
        int(
            (temp["STEMI"] == 1).sum()
        )
    )

    print(
        "NSTEMI:",
        int(
            (temp["NSTEMI"] == 1).sum()
        )
    )

    for vessel in vessel_cols:

        print(
            vessel,
            int(
                temp[vessel].sum()
            )
        )


# ============================================================
# MOST COMMON OMI PHENOTYPE COMBINATIONS
# ============================================================

omi["phenotype"] = (
    omi[
        [
            "STEMI",
            "NSTEMI",
            "LM_GROUP",
            "LAD_GROUP",
            "LCX_GROUP",
            "RCA_GROUP"
        ]
    ]
    .astype(int)
    .astype(str)
    .agg("-".join, axis=1)
)

print("\n================================================")
print("MOST COMMON OMI PHENOTYPES")
print("Format:")
print("STEMI-NSTEMI-LM-LAD-LCX-RCA")
print("================================================")

print(
    omi["phenotype"]
    .value_counts()
    .head(20)
)

OMI OVERVIEW
Total OMI: 1151
Train OMI: 985
Validation OMI: 166

OMI vs STEMI / NSTEMI
OMI + STEMI only              :  727 (63.16%)
OMI + NSTEMI only             :  423 (36.75%)
OMI + STEMI + NSTEMI          :    0 ( 0.00%)
OMI neither STEMI nor NSTEMI  :    1 ( 0.09%)

OMI BY MAJOR VESSEL
LM_GROUP  :    5 ( 0.43%)
LAD_GROUP :  467 (40.57%)
LCX_GROUP :  326 (28.32%)
RCA_GROUP :  544 (47.26%)

NUMBER OF VESSEL GROUPS PER OMI CASE
n_vessel_groups
0      6
1    966
2    161
3     18
Name: count, dtype: int64

OMI with NO major vessel label: 6 (0.52%)

OMI + CTO: 0 (0.00%)

TRAIN / VALIDATION OMI PHENOTYPES

--- TRAIN ---
OMI: 985
STEMI: 628
NSTEMI: 356
LM_GROUP 4
LAD_GROUP 401
LCX_GROUP 273
RCA_GROUP 461

--- VALIDATION ---
OMI: 166
STEMI: 99
NSTEMI: 67
LM_GROUP 1
LAD_GROUP 66
LCX_GROUP 53
RCA_GROUP 83

MOST COMMON OMI PHENOTYPES
Format:
STEMI-NSTEMI-LM-LAD-LCX-RCA
phenotype
1-0-0-0-0-1    262
1-0-0-1-0-0    253
0-1-0-0-0-1    140
0-1-0-0-1-0    110
1-0-0-0-1-0    101
0-1-0-1-0-0     95
